## Metrica Data Processing

In [1]:
import os

wd = os.path.normpath(os.getcwd() + '/..')
os.chdir(wd)
os.getcwd()

'/home/users/hyunsung/workspace/ballradar'

In [27]:
%load_ext autoreload
%autoreload 2

import json
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
from matplotlib import animation
from tqdm import tqdm

from dataset import SoccerDataset
from datatools.metrica_data import MetricaData
from datatools.animator import Animator
from datatools.trace_helper import TraceHelper
from models import load_model

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Parsing Metrica Sample Game 3 Data

In [18]:
tree = ET.parse("data/metrica/data/Sample_Game_3/Sample_Game_3_metadata.xml")
root = tree.getroot()
root[0].tag, root[1].tag

('Metadata', 'DataFormatSpecifications')

In [47]:
player_records = []
phase_records = []

for player in root.iter("Player"):
    team_code = player.get("teamId")[-1]
    squad_num = int(player.findtext("ShirtNumber"))
    player_code = f"{team_code}{squad_num:02d}"

    for param in player.iter("ProviderParameter"):
        if param.findtext("Name") == "position_type":
            position = param.findtext("Value")

    player_records.append([squad_num, player_code, position])

player_records = pd.DataFrame(player_records, columns=["squad_num", "code", "position"]).set_index("squad_num")

for i, data_spec in enumerate(root[1]):
    start_frame = int(data_spec.get("startFrame"))
    end_frame = int(data_spec.get("endFrame"))
    session = 1 if i == 0 else 2

    player_codes = []
    gk_codes = []

    for player_xy in data_spec[1]:
        squad_num = int(player_xy[0].get("playerChannelId")[6:-2])
        player_code = player_records.at[squad_num, "code"]
        player_codes.append(player_code)

        position = player_records.at[squad_num, "position"]
        if position == "Goalkeeper":
            gk_codes.append(player_code)
    
    player_codes = player_codes[10:11] + player_codes[:10] + player_codes[-1:] + player_codes[11:-1]
    phase_records.append([i + 1, session, start_frame, end_frame, player_codes, gk_codes])

header = ["phase", "session", "start_frame", "end_frame", "player_codes", "gk_codes"]
phase_records = pd.DataFrame(phase_records, columns=header).set_index("phase")
phase_records

,session,start_frame,end_frame,player_codes,gk_codes
phase,,,,,
1,1,1,69661,"[A11, A01, A02, A03, A04, A05, A06, A07, A08, ...","[A11, B28]"
2,2,69662,89697,"[A11, A01, A02, A12, A04, A05, A06, A07, A08, ...","[A11, B28]"
3,2,89698,93452,"[A11, A01, A02, A12, A04, A05, A06, A07, A08, ...","[A11, B28]"
4,2,93453,93835,"[A11, A01, A02, A12, A04, A05, A06, A07, A08, ...","[A11, B28]"
5,2,93836,94657,"[A11, A01, A02, A12, A04, A05, A06, A07, A08, ...","[A11, B28]"
6,2,94658,98472,"[A11, A01, A02, A12, A04, A05, A06, A07, A08, ...","[A11, B28]"
7,2,98473,102811,"[A11, A01, A02, A12, A04, A15, A06, A07, A08, ...","[A11, B28]"
8,2,102812,110298,"[A11, A01, A02, A12, A04, A15, A06, A07, A16, ...","[A11, B28]"
9,2,110299,120212,"[A11, A01, A02, A12, A04, A15, A06, A07, A16, ...",[A11]


In [48]:
time_cols = ["frame", "session", "time"]
xy_cols = np.array([[f"{p}_x", f"{p}_y"] for p in player_records["code"].tolist() + ["ball"]]).flatten().tolist()

traces_txt = pd.read_csv("data/metrica/data/Sample_Game_3/Sample_Game_3_tracking.txt", sep=";", header=None)
tracking = pd.DataFrame(index=traces_txt.index, columns=time_cols + xy_cols)

for phase in tqdm(phase_records.index):
    i0 = phase_records.at[phase, "start_frame"] - 1
    i1 = phase_records.at[phase, "end_frame"] - 1
    player_codes = phase_records.at[phase, "player_codes"]

    phase_traces = traces_txt.loc[i0:i1]
    phase_traces.columns = player_codes
    leftmost = phase_traces[player_codes[0]].str.split(":", expand=True)
    leftmost.columns = ["frame", player_codes[0]]
    rightmost = phase_traces[player_codes[-1]].str.split(":", expand=True)
    rightmost.columns = [player_codes[-1], "ball"]
    phase_traces = pd.concat([leftmost, phase_traces[player_codes[1:-1]], rightmost], axis=1)

    tracking.loc[phase_traces.index, "frame"] = phase_traces["frame"].astype(int)
    tracking.loc[phase_traces.index, "session"] = phase_records.at[phase, "session"]

    for p in phase_traces.columns[1:]:
        xy = phase_traces[p].str.split(",", expand=True).astype(float).values
        tracking.loc[phase_traces.index, [f"{p}_x", f"{p}_y"]] = xy

tracking["time"] = (tracking["frame"] * 0.04).astype(float).round(2)
tracking

100%|██████████| 11/11 [00:05<00:00,  2.01it/s]


,frame,session,time,A11_x,A11_y,A01_x,A01_y,A02_x,A02_y,A03_x,...,B32_x,B32_y,B33_x,B33_y,B34_x,B34_y,B35_x,B35_y,ball_x,ball_y
0,1,1,0.04,0.84722,0.52855,0.65268,0.24792,0.66525,0.46562,0.68103,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,0.08,0.84722,0.52855,0.65231,0.24513,0.66482,0.46548,0.68095,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,0.12,0.84722,0.52855,0.65197,0.24387,0.66467,0.46537,0.68078,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1,0.16,0.84722,0.52855,0.65166,0.24288,0.6646,0.46488,0.68063,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1,0.20,0.84722,0.52855,0.65141,0.24251,0.66452,0.46469,0.68052,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143756,143757,2,5750.28,0.11993,0.51783,0.47808,0.45408,NaN,NaN,NaN,...,0.80792,0.27106,0.73396,0.8533,0.90315,0.5375,0.50199,0.55081,NaN,NaN
143757,143758,2,5750.32,0.11993,0.51783,0.47786,0.45521,NaN,NaN,NaN,...,0.80712,0.27184,0.73251,0.85289,0.90301,0.53788,0.50164,0.55178,NaN,NaN
143758,143759,2,5750.36,0.11993,0.51783,0.47743,0.45709,NaN,NaN,NaN,...,0.80582,0.27242,0.73086,0.85218,0.90264,0.53799,0.50099,0.55329,NaN,NaN
143759,143760,2,5750.40,0.11993,0.51783,0.47669,0.45947,NaN,NaN,NaN,...,0.80444,0.2726,0.72892,0.85192,0.90204,0.53782,0.50003,0.55502,NaN,NaN


In [49]:
event_path = "data/metrica/data/Sample_Game_3/Sample_Game_3_events.json"
with open(event_path, "r", encoding="utf-8") as f:
    obj = json.load(f)

COL_DICT = {
    "team_name": "team",
    "type_name": "type",
    "subtypes_name": "subtype",
    "period": "period_id",
    "start_frame": "start_frame",
    "start_time": "start_time",
    "end_frame": "end_frame",
    "end_time": "end_time",
    "from_name": "from",
    "to_name": "to",
    "start_x": "start_x",
    "start_y": "start_y",
    "end_x": "end_x",
    "end_y": "end_y",
}
events = pd.json_normalize(obj["data"], sep="_")[COL_DICT.keys()].rename(columns=COL_DICT)
events

,team,type,subtype,period_id,start_frame,start_time,end_frame,end_time,from,to,start_x,start_y,end_x,end_y
0,Team A,SET PIECE,KICK OFF,1,361,14.44,361,14.44,Player 10,NaN,NaN,NaN,NaN,NaN
1,Team A,PASS,NaN,1,361,14.44,377,15.08,Player 10,Player 7,0.50125,0.48725,0.49864,0.48705
2,Team A,CARRY,NaN,1,377,15.08,384,15.36,Player 7,NaN,0.49864,0.48705,0.49700,0.48500
3,Team A,PASS,NaN,1,384,15.36,426,17.04,Player 7,Player 8,0.49700,0.48500,0.63373,0.63449
4,Team A,CARRY,NaN,1,426,17.04,465,18.60,Player 8,NaN,0.63373,0.63449,0.66986,0.59707
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3615,Team B,PASS,NaN,2,143406,5736.24,143469,5738.76,Player 33,Player 20,0.73416,0.40874,0.71353,0.85950
3616,Team B,PASS,NaN,2,143469,5738.76,143532,5741.28,Player 20,Player 28,0.71353,0.85950,0.88776,0.51189
3617,Team B,CARRY,NaN,2,143469,5738.76,143470,5738.80,Player 20,NaN,0.71353,0.85950,0.71353,0.85950
3618,Team B,CARRY,NaN,2,143532,5741.28,143553,5742.12,Player 28,NaN,0.88776,0.51189,0.89225,0.50456


In [50]:
events.to_csv(f"data/metrica/data/Sample_Game_3/Sample_Game_3_RawEventsData.csv", index=False)
tracking.to_csv(f"data/metrica/data/Sample_Game_3/Sample_Game_3_RawTrackingData.csv", index=False)

### Processing Metrica Data

In [37]:
match_id = 3

event_path = f"data/metrica/data/Sample_Game_{match_id}/Sample_Game_{match_id}_RawEventsData.csv"
events = pd.read_csv(event_path)

if match_id <= 2:
    home_path = f"data/metrica/data/Sample_Game_{match_id}/Sample_Game_{match_id}_RawTrackingData_Home_Team.csv"
    away_path = f"data/metrica/data/Sample_Game_{match_id}/Sample_Game_{match_id}_RawTrackingData_Away_Team.csv"
    home_tracking = pd.read_csv(home_path, header=[0, 1, 2])
    away_tracking = pd.read_csv(away_path, header=[0, 1, 2])
    helper = MetricaData(home_tracking, away_tracking, events=events)
else:  # match_id == 3
    trace_file = f"data/metrica/data/Sample_Game_{match_id}/Sample_Game_{match_id}_RawTrackingData.csv"
    tracking = pd.read_csv(trace_file, index_col=0)
    helper = MetricaData(tracking_from_txt=tracking, events=events)
    
helper.tracking

,period_id,timestamp,home_11_x,home_11_y,home_1_x,home_1_y,home_2_x,home_2_y,home_3_x,home_3_y,...,away_32_x,away_32_y,away_33_x,away_33_y,away_34_x,away_34_y,away_35_x,away_35_y,ball_x,ball_y
frame_id,,,,,,,,,,,,,,,,,,,,,
0,1,0.00,88.95810,35.94140,68.53140,16.85856,69.85125,31.66216,71.50815,40.17644,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,0.04,88.95810,35.94140,68.49255,16.66884,69.80610,31.65264,71.49975,40.15672,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,0.08,88.95810,35.94140,68.45685,16.58316,69.79035,31.64516,71.48190,40.14380,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,0.12,88.95810,35.94140,68.42430,16.51584,69.78300,31.61184,71.46615,40.11116,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,0.16,88.95810,35.94140,68.39805,16.49068,69.77460,31.59892,71.45460,40.07512,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143756,2,2963.80,12.59265,35.21244,50.19840,30.87744,NaN,NaN,NaN,NaN,...,84.8316,18.43208,77.06580,58.02440,94.83075,36.55000,52.70895,37.45508,NaN,NaN
143757,2,2963.84,12.59265,35.21244,50.17530,30.95428,NaN,NaN,NaN,NaN,...,84.7476,18.48512,76.91355,57.99652,94.81605,36.57584,52.67220,37.52104,NaN,NaN
143758,2,2963.88,12.59265,35.21244,50.13015,31.08212,NaN,NaN,NaN,NaN,...,84.6111,18.52456,76.74030,57.94824,94.77720,36.58332,52.60395,37.62372,NaN,NaN


In [ ]:
events, tracking = MetricaData.label_phases(helper.events, helper.tracking)
events, tracking = MetricaData.label_episodes(events, tracking)
events, tracking = MetricaData.label_possessions(events, tracking)

,frame_id,period_id,timestamp,phase_id,episode_id,ball_state,ball_owning_team_id,home_11_x,home_11_y,home_1_x,...,away_34_x,away_34_y,away_35_x,away_35_y,ball_x,ball_y,player_id,event_type,event_x,event_y
0,0,1,0.00,1,0,dead,home,88.95810,35.94140,68.53140,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52.63125,33.133
1,1,1,0.04,1,0,dead,home,88.95810,35.94140,68.49255,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52.63125,33.133
2,2,1,0.08,1,0,dead,home,88.95810,35.94140,68.45685,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52.63125,33.133
3,3,1,0.12,1,0,dead,home,88.95810,35.94140,68.42430,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52.63125,33.133
4,4,1,0.16,1,0,dead,home,88.95810,35.94140,68.39805,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52.63125,33.133
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144117,143756,2,2963.80,11,0,dead,away,12.59265,35.21244,50.19840,...,94.83075,36.55000,52.70895,37.45508,NaN,NaN,away_28,BALL LOST,76.84740,54.077
144118,143757,2,2963.84,11,0,dead,away,12.59265,35.21244,50.17530,...,94.81605,36.57584,52.67220,37.52104,NaN,NaN,away_28,BALL LOST,76.84740,54.077
144119,143758,2,2963.88,11,0,dead,away,12.59265,35.21244,50.13015,...,94.77720,36.58332,52.60395,37.62372,NaN,NaN,away_28,BALL LOST,76.84740,54.077
144120,143759,2,2963.92,11,0,dead,away,12.59265,35.21244,50.05245,...,94.71420,36.57176,52.50315,37.74136,NaN,NaN,away_28,BALL LOST,76.84740,54.077


In [ ]:
events.to_parquet(f"data/metrica/event/match{match_id}.parquet")
tracking.to_parquet(f"data/metrica/tracking/match{match_id}.parquet")

### Visualization for Metrica Data

##### Animating Trajectories

In [ ]:
merged = MetricaData.merge_for_animation(events, tracking)
merged[merged["period_id"] == 2]

,frame_id,period_id,timestamp,phase_id,episode_id,ball_state,ball_owning_team_id,home_11_x,home_11_y,home_1_x,...,away_34_x,away_34_y,away_35_x,away_35_y,ball_x,ball_y,player_id,event_type,event_x,event_y
69814,69661,2,0.00,2,0,dead,away,10.64910,34.29852,37.63935,...,NaN,NaN,NaN,NaN,NaN,NaN,away_28,BALL LOST,1.44375,31.01344
69815,69662,2,0.04,2,0,dead,away,10.64910,34.29852,37.53855,...,NaN,NaN,NaN,NaN,NaN,NaN,away_28,BALL LOST,1.44375,31.01344
69816,69663,2,0.08,2,0,dead,away,10.64910,34.29852,37.44090,...,NaN,NaN,NaN,NaN,NaN,NaN,away_28,BALL LOST,1.44375,31.01344
69817,69664,2,0.12,2,0,dead,away,10.64910,34.29852,37.33170,...,NaN,NaN,NaN,NaN,NaN,NaN,away_28,BALL LOST,1.44375,31.01344
69818,69665,2,0.16,2,0,dead,away,10.64910,34.29852,37.25190,...,NaN,NaN,NaN,NaN,NaN,NaN,away_28,BALL LOST,1.44375,31.01344
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144117,143756,2,2963.80,11,0,dead,away,12.59265,35.21244,50.19840,...,94.83075,36.55000,52.70895,37.45508,NaN,NaN,away_28,BALL LOST,76.84740,54.07700
144118,143757,2,2963.84,11,0,dead,away,12.59265,35.21244,50.17530,...,94.81605,36.57584,52.67220,37.52104,NaN,NaN,away_28,BALL LOST,76.84740,54.07700
144119,143758,2,2963.88,11,0,dead,away,12.59265,35.21244,50.13015,...,94.77720,36.58332,52.60395,37.62372,NaN,NaN,away_28,BALL LOST,76.84740,54.07700
144120,143759,2,2963.92,11,0,dead,away,12.59265,35.21244,50.05245,...,94.71420,36.57176,52.50315,37.74136,NaN,NaN,away_28,BALL LOST,76.84740,54.07700


In [42]:
f0, f1 = 69814, 69814 + 1500
track_dict = {"main": merged.loc[f0:f1]}

animator = Animator(track_dict, show_times=True, show_episodes=True, show_events=True)
anim = animator.run()

anim_path = f"animations/metrica_match{match_id}_{f0}-{f1}.mp4"
writer = animation.FFMpegWriter(fps=25)
os.makedirs("animations", exist_ok=True)
anim.save(anim_path, writer=writer)

##### Animating Feature Plots

In [ ]:
session = 1
tracking = helper.tracking[helper.tracking["session"] == session]
anim = TraceHelper.plot_speeds_and_accels(tracking, helper.home_players)
writer = animation.FFMpegWriter(fps=5)

smoothing = True
if smoothing:
    path = f"animations/feature_plots/metrica_match{match_id}_s{session}_smooth.mp4"
else:
    path = f"animations/feature_plots/metrica_match{match_id}_s{session}_noisy.mp4"
    
anim.save(path, writer=writer)